In [9]:
import pandas as pd
import re
from functools import reduce
df = pd.read_csv('etreproprio_rhone_propre.csv', encoding='utf-8-sig')
print(f"Nombre d'annonces : {len(df)}")
print(f"Colonnes : {list(df.columns)}")
df.head(3)

Nombre d'annonces : 578
Colonnes : ['titre', 'description', 'ville', 'type_bien', 'etage', 'prix', 'surface', 'pieces', 'DPE', 'full_description', 'lien']


,titre,description,ville,type_bien,etage,prix,surface,pieces,DPE,full_description,lien
0,Vente Appartement 72 m2 Francheville,Appartement 72 m2 à Francheville,francheville,Appartement,1,269000.0,72.0,3.0,D,Decultieux Immobililer la référence depuis 196...,https://www.etreproprio.com/immobilier-2529512...
1,Vente Appartement 78 m2 Villeurbanne,Appartement 78 m2 à Villeurbanne,villeurbanne,Appartement,3,293000.0,78.0,4.0,C,Une exclusivité DECULTIEUX Immobilier\nVILLEUR...,https://www.etreproprio.com/immobilier-2529512...
2,Vente Appartement 208 m2 Feyzin,EXCLUSIVITÉ Feyzin La Bégude Appartement r...,feyzin,Appartement,0,380000.0,208.0,5.0,A,FEYZIN LA BÉGUDE Le confort d’une maison réc...,https://www.etreproprio.com/immobilier-2529551...


In [8]:
bases_fusionnees = spark.read.csv("bases_fusionnees.csv", header=True, inferSchema=True, sep=",")

NameError: name 'spark' is not defined

In [ ]:
bases_fusionnees.columns

['type_bien', 'pieces', 'prix', 'surface', 'etage', 'ville', 'code_postal']

In [ ]:
print("=== Statistiques descriptives ===")
bases_fusionnees.describe().show()

print("=== Repartition par type de bien ===")
bases_fusionnees.groupBy("type_bien").count().orderBy(F.desc("count")).show()

print("=== Repartition par nombre de pieces ===")
bases_fusionnees.groupBy("pieces").count().orderBy("pieces").show()

print("=== Top 10 villes ===")
bases_fusionnees.groupBy("ville").count().orderBy(F.desc("count")).show(10)

=== Statistiques descriptives ===
+-------+-----------+------------------+-----------------+------------------+------------------+------+------------------+
|summary|  type_bien|            pieces|             prix|           surface|             etage| ville|       code_postal|
+-------+-----------+------------------+-----------------+------------------+------------------+------+------------------+
|  count|       8019|              8019|             8019|              8019|              8019|  8019|              8019|
|   mean|       NULL|  4.05100386581868|427315.1996508293|101.58756702830776|2.8797779201366644|  NULL|  69273.8709315376|
| stddev|       NULL|1.6509496507084573|300296.6495465243| 61.13174260486775|  2.33179621631308|  NULL|252.03671516019935|
|    min|appartement|               1.0|          49000.0|               9.0|                -1|affoux|             69001|
|    max|     maison|              12.0|        4160000.0|             900.0|               RDC| éveux|  

In [ ]:
# Encodage des colonnes categoriques
indexer_ville = StringIndexer(inputCol="ville",       outputCol="ville_index",  handleInvalid="keep")
indexer_type  = StringIndexer(inputCol="type_clean",  outputCol="type_index",   handleInvalid="keep")
indexer_etage = StringIndexer(inputCol="etage_clean", outputCol="etage_index",  handleInvalid="keep")

# Liste des features
features_cols = [
    "surface", "pieces", "chambres", "terrain", "neuf_flag",
    "ville_index", "type_index", "etage_index"
]

assembler = VectorAssembler(inputCols=features_cols, outputCol="features")

print("Features utilisees :")
for f in features_cols:
    print(f"  - {f}")

Features utilisees :
  - surface
  - pieces
  - chambres
  - terrain
  - neuf_flag
  - ville_index
  - type_index
  - etage_index


In [ ]:
train_data, test_data = bases_fusionnees.randomSplit([0.8, 0.2], seed=42)

train_data = train_data.cache()
test_data  = test_data.cache()

print(f"Taille train : {train_data.count()} lignes")
print(f"Taille test  : {test_data.count()} lignes")

Taille train : 6486 lignes
Taille test  : 1533 lignes


In [ ]:
bases_fusionnees.printSchema()
print(bases_fusionnees.columns)

root
 |-- type_bien: string (nullable = true)
 |-- pieces: double (nullable = true)
 |-- prix: double (nullable = true)
 |-- surface: double (nullable = true)
 |-- etage: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- code_postal: integer (nullable = true)

['type_bien', 'pieces', 'prix', 'surface', 'etage', 'ville', 'code_postal']


In [ ]:
train_data, test_data = bases_fusionnees.randomSplit([0.8, 0.2], seed=42)

In [ ]:
train_data

DataFrame[type_bien: string, pieces: double, prix: double, surface: double, etage: string, ville: string, code_postal: int]

In [ ]:
indexer_ville = StringIndexer(inputCol="ville", outputCol="ville_idx", handleInvalid="keep")
indexer_type  = StringIndexer(inputCol="type_bien", outputCol="type_idx", handleInvalid="keep")
indexer_etage = StringIndexer(inputCol="etage", outputCol="etage_idx", handleInvalid="keep")

assembler = VectorAssembler(
    inputCols=["type_idx", "pieces", "surface", "etage_idx", "ville_idx"],
    outputCol="features"
)

In [ ]:
dt = DecisionTreeRegressor(featuresCol="features", labelCol="prix", maxDepth=8,maxBins=256, seed=42)
pipeline_dt = Pipeline(stages=[indexer_ville, indexer_type, indexer_etage, assembler, dt])

t0 = time.perf_counter()
model_dt = pipeline_dt.fit(train_data)
dt_train_time = time.perf_counter() - t0
print(f"Decision Tree - temps entrainement : {dt_train_time:.2f} s")

Decision Tree - temps entrainement : 1.83 s


In [ ]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="prix",
    maxBins=256,
    numTrees=100,
    maxDepth=10,
    seed=42
)

pipeline_rf = Pipeline(stages=[indexer_ville, indexer_type, indexer_etage, assembler, rf])

t0 = time.perf_counter()
model_rf = pipeline_rf.fit(train_data)
rf_train_time = time.perf_counter() - t0

print(f"Random Forest - temps entrainement : {rf_train_time:.2f} s")

Random Forest - temps entrainement : 17.70 s


In [ ]:
gbt = GBTRegressor(
    featuresCol="features",
    labelCol="prix",
    maxIter=100,
    maxDepth=8,
    maxBins=256,
    stepSize=0.1,
    seed=42
)

pipeline_gbt = Pipeline(stages=[indexer_ville, indexer_type, indexer_etage, assembler, gbt])

t0 = time.perf_counter()
model_gbt = pipeline_gbt.fit(train_data)
gbt_train_time = time.perf_counter() - t0

print(f"GBT/XGBoost - temps entrainement : {gbt_train_time:.2f} s")

GBT/XGBoost - temps entrainement : 110.14 s


In [ ]:
# Prédictions sur le jeu de test
dt_pred  = model_dt.transform(test_data)
rf_pred  = model_rf.transform(test_data)
gbt_pred = model_gbt.transform(test_data)

# Evaluateurs
r2_eval   = RegressionEvaluator(labelCol="prix", predictionCol="prediction", metricName="r2")
rmse_eval = RegressionEvaluator(labelCol="prix", predictionCol="prediction", metricName="rmse")
mae_eval  = RegressionEvaluator(labelCol="prix", predictionCol="prediction", metricName="mae")

# Calcul des métriques
dt_r2,  dt_rmse,  dt_mae  = r2_eval.evaluate(dt_pred),  rmse_eval.evaluate(dt_pred),  mae_eval.evaluate(dt_pred)
rf_r2,  rf_rmse,  rf_mae  = r2_eval.evaluate(rf_pred),  rmse_eval.evaluate(rf_pred),  mae_eval.evaluate(rf_pred)
gbt_r2, gbt_rmse, gbt_mae = r2_eval.evaluate(gbt_pred), rmse_eval.evaluate(gbt_pred), mae_eval.evaluate(gbt_pred)

# Tableau comparatif
sep = "=" * 72
print(sep)
print(f"{'Metrique':<20} {'Decision Tree':>15} {'Random Forest':>15} {'GBT/XGBoost':>15}")
print(sep)
print(f"{'R2':<20} {dt_r2:>15.4f} {rf_r2:>15.4f} {gbt_r2:>15.4f}")
print(f"{'RMSE (euros)':<20} {dt_rmse:>15,.0f} {rf_rmse:>15,.0f} {gbt_rmse:>15,.0f}")
print(f"{'MAE (euros)':<20} {dt_mae:>15,.0f} {rf_mae:>15,.0f} {gbt_mae:>15,.0f}")
print(f"{'Temps train (s)':<20} {dt_train_time:>15.2f} {rf_train_time:>15.2f} {gbt_train_time:>15.2f}")
print(sep)

scores = {"Decision Tree": dt_r2, "Random Forest": rf_r2, "GBT/XGBoost": gbt_r2}
best   = max(scores, key=scores.get)
print(f"\n=> Meilleur modele : {best} (R2 = {scores[best]:.4f})")

Metrique               Decision Tree   Random Forest     GBT/XGBoost
R2                            0.5761          0.6408          0.5112
RMSE (euros)                 200,175         184,267         214,940
MAE (euros)                  105,386         106,517         113,177
Temps train (s)                 1.83           17.70          110.14

=> Meilleur modele : Random Forest (R2 = 0.6408)
